# Checkpoint and reload a Multi30k translator

Trains a `Multi30kTranslator` for one epoch with `fit`, then loads the checkpoint
`fit` wrote into a freshly built model and checks that it is the trained model: the
weights match, the optimizer and scheduler state come back, and a validation pass
over the reloaded model gives `fit`'s val loss.

`fit` calls `checkpoint` after every validation pass and writes
`app/history/checkpoints/<run-uuid>/epoch_N.pt`. The file holds `state_dict`s only,
so `load` needs a `TrainState` built from the same config: same model, optimizer,
and scheduler.

Run it like [hello_world.ipynb](hello_world.ipynb), whose path cell is copied below:

```bash
uv run jupyter nbconvert --execute --to notebook --inplace \
    app/src/notebooks/multi30k_checkpoint.ipynb
```

In [1]:
import sys
from pathlib import Path

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "app" / "src").is_dir()
)
sys.path.insert(0, str(repo_root / "app" / "src"))  # noqa: NAR001

print(f"imports resolve from {repo_root / 'app' / 'src'}")  # noqa: NAR001

imports resolve from /Users/anantsimran/code/dawnshard/app/src


## A small config

The translator's own YAML, shrunk so one epoch is quick: 128 wide and two blocks a
side instead of 512 and six. The data settings stay as they are, so the BPE
tokenizers this trains (or loads) are the ones a full run uses. `train_seed` is
pinned so the run repeats, and `run_uuid` names the checkpoint folder.

In [2]:
from uuid import uuid4

import torch
from training.constants import ATTENTION_PROBES_PATH, CHECKPOINTS_PATH
from training.dal.dao.sqlite_dao import SqliteDao
from training.dataload.constants import CACHE_PATH
from training.run.multi30k_translator import (
    DEFAULT_CONFIG_PATH,
    Multi30kTranslatorTrainConfig,
    build_epoch_spec,
    build_loaders,
    build_train_state,
    loader_seq_len,
)
from training.train.checkpoint import load
from training.train.train_loop import eval_step, fit, run_epoch

run_uuid = str(uuid4())  # noqa: NAR001
config = Multi30kTranslatorTrainConfig.from_yaml(path=DEFAULT_CONFIG_PATH).with_overrides(
    overrides={
        "model": {"d_model": 128, "num_encoders": 2, "num_decoders": 2},
        "fit": {"num_epochs": 1, "train_seed": 0, "run_uuid": run_uuid},
    }
)
print(config.model)  # noqa: NAR001
print(config.fit)  # noqa: NAR001

TransformerConfig(d_model=128, src=EmbeddingConfig(seq_length=32, dropout=0.3, position_encoding=True), tgt=EmbeddingConfig(seq_length=32, dropout=0.3, position_encoding=True), num_encoders=2, num_decoders=2, encoder=BlockConfig(attention=AttentionConfig(h=4, qk_norm=False), ffn_multiplier=6, dropout=0.3, norm='layernorm'), decoder=BlockConfig(attention=AttentionConfig(h=4, qk_norm=False), ffn_multiplier=6, dropout=0.3, norm='layernorm'))
FitConfig(num_epochs=1, val_seed=1000000, test_seed=2000000, train_seed=0, val_epoch_list=None, epoch_start=1, run_uuid='ad2ec7fe-b54f-46da-afad-b06db7fc2868', early_stopping=None)


In [3]:
with SqliteDao(path=CACHE_PATH) as cache:
    train_loader, val_loader, tokenizers = build_loaders(
        dataload_config=config.data,
        seq_len=loader_seq_len(model_config=config.model),
        batch_size=config.dataloader.batch_size,
        cache=cache,
    )
src_vocab_size, tgt_vocab_size = (len(tokenizer.vocab) for tokenizer in tokenizers)  # noqa: NAR001
print(f"{len(train_loader)} train batches, {len(val_loader)} val batches")  # noqa: NAR001

2026-11-05 21:56:05.161 | INFO     | training.dal.cache:wrapper:121 - cache hit: training.dataload.multi30k.get_multi30k_texts (e8f1e7603a7f)


2026-11-05 21:56:05.352 | INFO     | training.utils.timing:log_elapsed:21 - BPE trained merges for multi30k-en-10000.json in 0.2s


2026-11-05 21:56:05.360 | INFO     | training.dal.cache:wrapper:121 - cache hit: training.dataload.multi30k.get_multi30k_texts (0ca29f86f145)


2026-11-05 21:56:05.802 | INFO     | training.utils.timing:log_elapsed:21 - BPE trained merges for multi30k-de-10000.json in 0.4s


2026-11-05 21:56:05.806 | INFO     | training.dal.cache:wrapper:121 - cache hit: training.dataload.multi30k.get_multi30k_texts (e8f1e7603a7f)


2026-11-05 21:56:05.809 | INFO     | training.dal.cache:wrapper:121 - cache hit: training.dataload.multi30k.get_multi30k_texts (0ca29f86f145)


... (3 outputs left out)

2026-11-05 21:56:08.930 | INFO     | training.dal.cache:wrapper:124 - cache miss: training.preprocess.bpe.encode._encode_unique (be58d5207aad)


2026-11-05 21:56:09.001 | INFO     | training.dal.cache:wrapper:124 - cache miss: training.preprocess.bpe.encode._encode_unique (2c4e84e948cf)


2026-11-05 21:56:09.046 | INFO     | training.dal.cache:wrapper:124 - cache miss: training.preprocess.bpe.encode._encode_unique (b42933dc34f2)


454 train batches, 16 val batches


## Train one epoch

`fit` validates after epoch 1 (the YAML leaves `val_epoch_list` out, so every epoch
validates), then writes the checkpoint.

In [4]:
torch.manual_seed(seed=config.fit.train_seed)
state = build_train_state(
    config=config, src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size
)
epoch_spec = build_epoch_spec(probe_dir=ATTENTION_PROBES_PATH / run_uuid, num_sentences=2)
history = fit(
    state=state,
    epoch_spec=epoch_spec,
    train_loader=train_loader,
    val_loader=val_loader,
    fit_config=config.fit,
)

checkpoint_path = CHECKPOINTS_PATH / run_uuid / "epoch_1.pt"
fit_val_loss = history[-1].val_loss
print(f"val loss after epoch 1: {fit_val_loss:.6f}")  # noqa: NAR001
print(f"checkpoint: {checkpoint_path} ({checkpoint_path.stat().st_size / 1e6:.1f} MB)")  # noqa: NAR001

2026-11-05 21:56:09.612 | INFO     | training.train.train_loop:fit:190 - fit | epochs 1-1 | val_epochs None | seeds 0/1000000/2000000 | device mps | wandb False | run ad2ec7fe-b54f-46da-afad-b06db7fc2868


2026-11-05 21:56:31.894 | INFO     | training.train.train_loop:fit:237 - epoch   1 | train 4.8017 | val 3.9699


2026-11-05 21:56:31.996 | INFO     | training.train.checkpoint:checkpoint:39 - saved checkpoint to /Users/anantsimran/code/dawnshard/app/history/checkpoints/ad2ec7fe-b54f-46da-afad-b06db7fc2868/epoch_1.pt


val loss after epoch 1: 3.969938
checkpoint: /Users/anantsimran/code/dawnshard/app/history/checkpoints/ad2ec7fe-b54f-46da-afad-b06db7fc2868/epoch_1.pt (61.7 MB)


## Load it into a fresh model

A different seed gives the fresh model different weights, so a match after `load`
can only come from the file.

In [5]:
torch.manual_seed(seed=1)
restored = build_train_state(
    config=config, src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size
)

trained_weights = state.model.state_dict()
print(  # noqa: NAR001
    "same weights before load:",
    all(  # noqa: NAR001
        torch.equal(input=trained_weights[name], other=tensor)
        for name, tensor in restored.model.state_dict().items()
    ),
)

load(state=restored, epoch_spec=epoch_spec, checkpoint_path=checkpoint_path)
print(  # noqa: NAR001
    "same weights after load: ",
    all(  # noqa: NAR001
        torch.equal(input=trained_weights[name], other=tensor)
        for name, tensor in restored.model.state_dict().items()
    ),
)

same weights before load: False
same weights after load:  True


The optimizer and scheduler come back too, which is what lets a run resume at
epoch 2 rather than restart: Adam's step count equals the number of training
batches, and the cosine schedule is one epoch in. The schedule is sized for this
config's `num_epochs=1`, so one epoch in is the end of the cosine and the learning
rate reads 0.

In [6]:
adam_steps = {
    int(param_state["step"])  # noqa: NAR001
    for param_state in restored.optimizer.state_dict()["state"].values()
}
print(f"Adam steps: {adam_steps}, train batches: {len(train_loader)}")  # noqa: NAR001
print(  # noqa: NAR001
    f"scheduler epoch: {restored.scheduler.last_epoch}, lr: {restored.scheduler.get_last_lr()}"
)

Adam steps: {454}, train batches: 454
scheduler epoch: 1, lr: [0.0]


## Validate the reloaded model

The same validation pass `fit` ran, now over the reloaded model. Equal losses mean
the file holds the trained translator, not just tensors of the right shape.

In [7]:
val_metrics = run_epoch(
    state=restored,
    epoch_spec=epoch_spec,
    loader=val_loader,
    step_fn=eval_step,
    epoch=1,
    train=False,
    seed=config.fit.val_seed,
)
print(f"val loss from fit:            {fit_val_loss:.6f}")  # noqa: NAR001
print(f"val loss of reloaded model:   {val_metrics.loss:.6f}")  # noqa: NAR001
print(f"val accuracy of reloaded model: {val_metrics.accuracy:.4f}")  # noqa: NAR001

val loss from fit:            3.969938
val loss of reloaded model:   3.969938
val accuracy of reloaded model: 0.3952
